# Ch. 8 · Notebook de référence — une étude reproductible de bout en bout

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch08-projet-integrateur.ipynb)

**IA pour les sciences — M1.** Cas réel : diagnostic de tumeurs du sein (Wisconsin), de la question à la fiche de modèle.

**Durée indicative :** 2 h (TP) + travail de projet

**Objectifs**

- Formuler une question et un critère de réussite avant toute modélisation.
- Décrire les données et leurs limites.
- Comparer baseline et modèles par validation croisée, puis évaluer une seule fois sur le test.
- Choisir un seuil de décision selon le coût des erreurs.
- Rapporter incertitude, analyse d’erreurs, importance des variables et environnement logiciel.

**Prérequis :** chapitres 2, 3 et 7. Servez-vous de ce notebook comme **gabarit** pour votre propre projet.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Question, hypothèse, critère de réussite

- **Question.** Des descripteurs morphologiques de noyaux cellulaires (images d’aspiration à l’aiguille fine) permettent-ils de distinguer tumeurs malignes et bénignes ?
- **Unité d’observation.** Une tumeur (une patiente).
- **Critère de réussite fixé à l’avance.** Rappel des tumeurs malignes ≥ 0,95 (un faux négatif est grave), avec la meilleure précision possible.
- **Réponse négative possible.** Si aucun modèle n’atteint ce rappel avec une précision acceptable, la conclusion est que ces descripteurs ne suffisent pas.

In [ ]:
import sys
import warnings
import numpy as np
warnings.filterwarnings('ignore', category=UserWarning)
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

GRAINE = 42
data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)     # 1 = maligne (la classe à détecter)
print(X.shape, '— malignes :', int(y.sum()), '| bénignes :', int((1 - y).sum()))
print('Valeurs manquantes :', int(X.isna().sum().sum()))
display(X.iloc[:, :10].describe().T[['mean', 'std', 'min', 'max']].round(2))

## 2. Limites des données (à écrire avant de modéliser)

569 cas d’un seul centre (Wisconsin, années 1990) ; 30 descripteurs calculés par un logiciel à partir d’images (moyenne, erreur standard
et « pire » valeur de 10 mesures) : ils sont **fortement corrélés** entre eux. La population, l’appareil et la procédure d’annotation
ne sont pas représentatifs d’un autre hôpital : toute conclusion est **interne** à ce jeu.

## 3. Protocole : test réservé, validation croisée sur l’entraînement

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=GRAINE)
cv = StratifiedKFold(5, shuffle=True, random_state=GRAINE)
modeles = {
    'Baseline (majoritaire)': DummyClassifier(strategy='most_frequent'),
    'Logistique': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                LogisticRegression(max_iter=5000)),
    'Forêt aléatoire': make_pipeline(SimpleImputer(strategy='median'),
                                     RandomForestClassifier(n_estimators=300, random_state=GRAINE)),
}
lignes = []
for nom, m in modeles.items():
    r = cross_validate(m, X_tr, y_tr, cv=cv, scoring=['balanced_accuracy', 'recall', 'precision', 'roc_auc'])
    lignes.append({'modèle': nom, **{k[5:]: f"{v.mean():.3f} ± {v.std():.3f}" for k, v in r.items() if k.startswith('test_')}})
print(pd.DataFrame(lignes).to_string(index=False))

## 4. Choisir le seuil selon le coût des erreurs (sur des prédictions de validation)

Le seuil par défaut 0,5 n’a rien de scientifique. On obtient des probabilités **hors échantillon** par validation croisée
(`cross_val_predict`) sur l’entraînement, puis on choisit le **plus grand seuil** qui garantit un rappel ≥ 0,95.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

modele = modeles['Logistique']
proba_cv = cross_val_predict(modele, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
precision, rappel, seuils = precision_recall_curve(y_tr, proba_cv)
ok = rappel[:-1] >= 0.95
seuil = seuils[ok].max()
print(f'Seuil retenu : {seuil:.3f} (rappel CV {rappel[:-1][ok][-1]:.3f}, précision CV {precision[:-1][ok][-1]:.3f})')
plt.plot(seuils, precision[:-1], label='précision'); plt.plot(seuils, rappel[:-1], label='rappel')
plt.axvline(seuil, color='k', ls=':', label='seuil retenu'); plt.axhline(0.95, color='gray', ls='--')
plt.xlabel('Seuil sur la probabilité de malignité'); plt.legend(); plt.title('Compromis précision–rappel (validation croisée)'); plt.show()

## 5. Évaluation finale, une seule fois, avec incertitude

In [ ]:
from sklearn.metrics import recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay

modele.fit(X_tr, y_tr)
proba_te = modele.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= seuil).astype(int)

def ic_bootstrap(y_true, y_pred, f, B=2000, graine=GRAINE):
    r = np.random.default_rng(graine)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    vals = [f(y_true[i], y_pred[i]) for i in (r.integers(0, len(y_true), len(y_true)) for _ in range(B))]
    return np.percentile(vals, [2.5, 97.5])

for nom, f in [('rappel', recall_score), ('précision', precision_score)]:
    bas, haut = ic_bootstrap(y_te, pred_te, f)
    print(f'{nom:9s} test = {f(y_te, pred_te):.3f}   IC95 % [{bas:.3f}, {haut:.3f}]')
ConfusionMatrixDisplay(confusion_matrix(y_te, pred_te), display_labels=['bénigne', 'maligne']).plot(cmap='Blues')
plt.title(f'Test (n={len(y_te)}), seuil {seuil:.2f}'); plt.show()

## 6. Analyse d’erreurs et importance des variables

On examine les cas mal classés et l’importance par permutation. Rappel du chapitre 7 : l’importance décrit le **modèle**,
pas une cause biologique ; avec des variables très corrélées, elle se « répartit » entre variables redondantes.

In [ ]:
from sklearn.inspection import permutation_importance

erreurs = X_te[pred_te != y_te].assign(vraie_classe=y_te[pred_te != y_te], proba=proba_te[pred_te != y_te].round(3))
print(len(erreurs), 'erreur(s) :'); display(erreurs[['mean radius', 'mean concave points', 'worst area', 'vraie_classe', 'proba']])

imp = permutation_importance(modele, X_te, y_te, scoring='recall', n_repeats=30, random_state=GRAINE)
top = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(8)
top[::-1].plot.barh(); plt.xlabel('Baisse du rappel après permutation'); plt.title('Importance par permutation (test)'); plt.show()

## 7. Reproductibilité : l’environnement

In [ ]:
print('Python      :', sys.version.split()[0])
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('Graine      :', GRAINE)

### ✏️ Exercice 1 — Comparer au modèle forêt avec le même protocole

Répétez les étapes 4 et 5 avec la forêt aléatoire (seuil choisi par validation croisée pour un rappel ≥ 0,95). Quel modèle retiendriez-vous ? Les intervalles se chevauchent-ils ?

In [ ]:
foret = modeles['Forêt aléatoire']
p_cv = cross_val_predict(foret, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
pr, rc, th = precision_recall_curve(y_tr, p_cv)
seuil_f = ...
foret.fit(X_tr, y_tr)
pred_f = ...
for nom, f in [('rappel', recall_score), ('précision', precision_score)]:
    print(nom, round(f(y_te, pred_f), 3), ic_bootstrap(y_te, pred_f, f).round(3))

<details><summary><b>Voir le corrigé</b></summary>

```python
foret = modeles['Forêt aléatoire']
p_cv = cross_val_predict(foret, X_tr, y_tr, cv=cv, method='predict_proba')[:, 1]
pr, rc, th = precision_recall_curve(y_tr, p_cv)
seuil_f = th[rc[:-1] >= 0.95].max()
foret.fit(X_tr, y_tr)
pred_f = (foret.predict_proba(X_te)[:, 1] >= seuil_f).astype(int)
for nom, f in [('rappel', recall_score), ('précision', precision_score)]:
    print(nom, round(f(y_te, pred_f), 3), ic_bootstrap(y_te, pred_f, f).round(3))
```

Avec 143 cas de test, les intervalles des deux modèles se chevauchent largement : la différence n’est pas démontrée. À performance comparable, on peut préférer le modèle le plus simple et le mieux calibré.

</details>

## 8. Fiche de modèle (à compléter pour votre projet)

| Rubrique | Contenu |
|---|---|
| Question et usage prévu | … |
| Données (source, période, effectif, unité d’observation) | … |
| Prétraitements (dans le pipeline) | … |
| Protocole (découpage, validation, graine) | … |
| Modèles comparés et baseline | … |
| Résultats test avec IC 95 % | … |
| Seuil et justification | … |
| Analyse d’erreurs | … |
| Limites et usages à proscrire | … |
| Usage d’outils d’IA générative dans le projet | … |

## Interprétation scientifique (à rédiger)

1. Le critère fixé au départ est-il atteint ? Avec quelle incertitude ?
2. Quelles erreurs restent, et quelles seraient leurs conséquences ?
3. Pourquoi ce résultat ne suffit-il pas pour un usage clinique ? Quelle étude faudrait-il ensuite ?
4. Qu’est-ce qui, dans ce notebook, permet à un collègue de reproduire exactement vos chiffres ?